In [ ]:
"""
Final Project - 3D Voxel Classification Pipeline
"""

# Imports
import numpy as np
import pandas as pd
import laspy
import open3d as o3d
from sklearn.decomposition import PCA
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt


In [ ]:
# Load point cloud and remove outliers
print("Load and denoise point cloud")

input_file = "final_project_subset.laz"
las = laspy.read(input_file)
points = np.vstack((las.x, las.y, las.z)).T
print(f"Loaded {len(points):,} points from {input_file}")

# Denoising with Open3D
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd, ind = pcd.remove_statistical_outlier(nb_neighbors=50, std_ratio=5.0)
points_filtered = np.asarray(pcd.points)
print(f"Removed {len(points) - len(points_filtered):,} outliers → {len(points_filtered):,} remaining")

np.save("denoised_subset_points.npy", points_filtered)

In [ ]:
# Voxelization
print("Voxelization:")

voxel_size = 2.0
min_coords = np.floor(points_filtered.min(axis=0))
max_coords = np.ceil(points_filtered.max(axis=0))
dims = np.ceil((max_coords - min_coords) / voxel_size).astype(int)

coords = np.floor((points_filtered - min_coords) / voxel_size).astype(int)
coords = np.clip(coords, 0, dims - 1)
voxel_ids_all = np.ravel_multi_index((coords[:, 0], coords[:, 1], coords[:, 2]), dims=dims)

unique_ids, counts = np.unique(voxel_ids_all, return_counts=True)
valid_ids = unique_ids[counts >= 10]
mask_valid = np.isin(voxel_ids_all, valid_ids)
voxel_points = points_filtered[mask_valid]
voxel_ids = voxel_ids_all[mask_valid]

print(f"Kept {len(voxel_points):,} points in {len(valid_ids):,} voxels")

np.save("voxel_ids.npy", voxel_ids)
np.save("voxel_points.npy", voxel_points)


In [ ]:
# PCA feature extraction
print("Feature extraction per voxel using PCA:")

voxel_features = []
voxel_normals = {}

for voxel_id in valid_ids:
    pts = voxel_points[voxel_ids == voxel_id]
    
    # PCA on points in voxel
    pca = PCA(n_components=3).fit(pts)
    λ1, λ2, λ3 = np.sort(pca.explained_variance_)[::-1]
    
    # Handle degenerate voxels
    if λ1 == 0 and λ2 == 0 and λ3 == 0:
        λ1, λ2, λ3 = 1e-6, 1e-6, 1e-6  
    else:
        λ1 = max(λ1, 1e-6)
        λ2 = max(λ2, 1e-6)
        λ3 = max(λ3, 1e-6)
        
    # Normal vector
    n = pca.components_[-1]
    if n[2] < 0:
        n = -n
    voxel_normals[voxel_id] = n

    # Store features
    voxel_features.append({
        "voxel_id": voxel_id,
        "n_points": len(pts),
        "linearity": (λ1 - λ2) / λ1,
        "planarity": (λ2 - λ3) / λ1,
        "scattering": λ3 / λ1,
        "omnivariance": (λ1 * λ2 * λ3) ** (1/3),
        "sum_ev": λ1 + λ2 + λ3,
        "anisotropy": (λ1 - λ3) / λ1,
        "eigentropy": -(λ1*np.log(λ1) + λ2*np.log(λ2) + λ3*np.log(λ3)),
        "change_curvature": λ3 / (λ1 + λ2 + λ3),
        "z_range": pts[:, 2].max() - pts[:, 2].min(),
        "plane_std": np.std(pca.transform(pts)[:, 2])
    })

df_features = pd.DataFrame(voxel_features)
print(f"Extracted {len(df_features)} voxels with PCA features")


In [ ]:
# Voxel center computation
print("Voxel center computation:")
voxel_centers = np.array([
    voxel_points[voxel_ids == vid].mean(axis=0)
    for vid in df_features["voxel_id"].values
])
np.save("voxel_centers.npy", voxel_centers)
print(f"Saved {len(voxel_centers)} voxel centers → voxel_centers.npy")

In [ ]:
# Spatial geometric features
print("Spatial geometric features extraction: 2D z-range, density ratio, curvature")

# KD-trees for 2D and 3D neighborhoods
points_tree_2d = cKDTree(voxel_points[:, :2])
points_tree_3d = cKDTree(voxel_points)
voxel_tree = cKDTree(voxel_centers)

# Neighborhood radii
R2D = voxel_size * 2.0   
R3D = voxel_size * 2.0   
Rcurv = voxel_size * 4.0 

zrange2d_list = []
density_ratio_list = []
geom_curv_list = []

for i, c in enumerate(voxel_centers):
    # 2D neighborhood (XY plane)
    idx2d = points_tree_2d.query_ball_point(c[:2], r=R2D)
    if len(idx2d) > 0:
        z_vals = voxel_points[idx2d, 2]
        zrange2d = z_vals.max() - z_vals.min()
    else:
        zrange2d = 0.0

    # 3D neighborhood (sphere)
    idx3d = points_tree_3d.query_ball_point(c, r=R3D)
    n2d = len(idx2d)
    n3d = len(idx3d)
    density_ratio = n3d / (n2d + 1e-6)

    # Geometric curvature: difference of normals with neighboring voxels
    curv_neighbors = voxel_tree.query_ball_point(c, r=Rcurv)
    if len(curv_neighbors) > 1:
        n_i = voxel_normals[df_features["voxel_id"].iloc[i]]
        diffs = []
        for j in curv_neighbors:
            n_j = voxel_normals[df_features["voxel_id"].iloc[j]]
            diffs.append(np.linalg.norm(n_i - n_j))
        geom_curv = np.mean(diffs)
    else:
        geom_curv = 0.0

    # Store features
    zrange2d_list.append(zrange2d)
    density_ratio_list.append(density_ratio)
    geom_curv_list.append(geom_curv)

# Add features
df_features["z_range_2d"] = zrange2d_list
df_features["density_ratio_3d_2d"] = density_ratio_list
df_features["geom_curvature"] = geom_curv_list

print(f"Added features: z_range_2d, density_ratio_3d_2d, geom_curvature for {len(df_features)} voxels")


In [ ]:
# Feature distribution visualization 

print("Plotting feature distributions...")

feature_names = [
    "linearity", "planarity", "scattering", "omnivariance", "sum_ev", "anisotropy",
    "eigentropy", "change_curvature", "z_range", "plane_std",
    "z_range_2d", "density_ratio_3d_2d", "geom_curvature"
]

ncols = 3
nrows = int(np.ceil(len(feature_names) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()

for i, feature in enumerate(feature_names):
    df_features[feature].hist(bins=30, ax=axes[i], color="blue")
    axes[i].set_title(feature, fontsize=18)
    axes[i].set_xlabel("Value", fontsize=16)
    axes[i].set_ylabel("Count", fontsize=16)
    axes[i].tick_params(axis='both', which='major', labelsize=16)

for j in range(len(feature_names), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()



In [ ]:
# Manual extraction
print("Manual extraction:")

def read_las_coords(filepath):
    las = laspy.read(filepath)
    return np.vstack((las.x, las.y, las.z)).T

label_files = {
    "building": "voxel_centers_building.las",
    "tree": "voxel_centers_trees.las",
    "terrain": "voxel_centers_terrain.las"
}

tree = cKDTree(voxel_centers)
df_features["label"] = "unlabeled"

# Assign labels using nearest voxel centers
for label, filepath in label_files.items():
    coords = read_las_coords(filepath)
    nearest_ids = df_features["voxel_id"].values[tree.query(coords, k=1)[1]]
    df_features.loc[df_features["voxel_id"].isin(nearest_ids), "label"] = label

print("\nLabel counts after sampling:")
print(df_features["label"].value_counts())

# Randomly keep 200 samples per labeled class
rng = np.random.default_rng(42)
for label in label_files.keys():
    idx = df_features.index[df_features["label"] == label]
    if len(idx) > 200:
        keep = rng.choice(idx, size=200, replace=False)
        drop = idx.difference(keep)
        df_features.loc[drop, "label"] = "unlabeled"

print("\nLabel counts after sampling:")
print(df_features["label"].value_counts())


In [ ]:
# Train CatBoost model

print("Train CatBoost model")

df = df_features.copy()

# Log-transformed features
df["scattering_log2"] = np.log2(df["scattering"] + 1e-6)
df["plane_std_log2"] = np.log2(df["plane_std"] + 1e-6)
df["z_range_log2"] = np.log2(df["z_range"] + 1e-6)
df["density_ratio_log2"] = np.log2(df["density_ratio_3d_2d"] + 1e-6)
df["geom_curvature_log2"] = np.log2(df["geom_curvature"] + 1e-6)

feature_cols = [
    "linearity", "planarity", "scattering_log2", "omnivariance", "sum_ev",
    "anisotropy", "eigentropy", "change_curvature",
    "z_range", "plane_std_log2",
    "z_range_2d", "density_ratio_3d_2d", "geom_curvature"
]

train_df = df[df["label"] != "unlabeled"]
X, y = train_df[feature_cols], train_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = CatBoostClassifier(
    iterations=300, learning_rate=0.1, depth=6,
    loss_function='MultiClass', verbose=100
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("\nClassification report:")
print(classification_report(y_test, y_pred))


# Feature importance visualization
importances = model.get_feature_importance()
feature_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(8, 4))
plt.barh(feature_importance["Feature"], feature_importance["Importance"], color="blue")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Classification of all voxels
print("Classification of all voxels:")

df["pred_label"] = model.predict(df[feature_cols]).flatten()
print(df["pred_label"].value_counts())

color_map = {
    "building": [1, 0, 0],
    "tree": [0, 1, 0],
    "terrain": [0.5, 0.25, 0],
    "unlabeled": [0.7, 0.7, 0.7]
}

#.png
colors_voxel = np.array([color_map.get(lbl, [0.6, 0.6, 0.6]) for lbl in df["pred_label"].values])
pcd_vox = o3d.geometry.PointCloud()
pcd_vox.points = o3d.utility.Vector3dVector(voxel_centers)
pcd_vox.colors = o3d.utility.Vector3dVector(colors_voxel)

vis = o3d.visualization.Visualizer()
vis.create_window(visible=False, width=1920, height=1080)
vis.add_geometry(pcd_vox)
vis.get_render_option().background_color = np.array([1, 1, 1])
vis.poll_events()
vis.update_renderer()
vis.capture_screen_image("classified_voxels.png", do_render=True)
vis.destroy_window()
print("Saved voxel classification → classified_voxels.png")


#.laz
rgb_colors = (colors_voxel * 255).astype(np.uint16)

header = laspy.LasHeader(point_format=3, version="1.2")
header.add_extra_dim(laspy.ExtraBytesParams(name="class_label", type=np.int8))

# Create LAS file with voxel centers
las_vox = laspy.LasData(header)
las_vox.x = voxel_centers[:, 0]
las_vox.y = voxel_centers[:, 1]
las_vox.z = voxel_centers[:, 2]
las_vox.red = rgb_colors[:, 0]
las_vox.green = rgb_colors[:, 1]
las_vox.blue = rgb_colors[:, 2]

label_map = {"building": 1, "tree": 2, "terrain": 3, "unlabeled": 0}
las_vox.class_label = np.array([label_map.get(lbl, 0) for lbl in df["pred_label"]])

las_vox.write("classified_voxels.laz")
print("Saved classified voxels → classified_voxels.laz")